# 01 — Exploratory Data Analysis

Explores the raw NH cost-report data (2015–2021) before any preprocessing.
Run **02_preprocessing.ipynb** next to produce .

## 1. Imports & configuration

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

warnings.filterwarnings("ignore", category=FutureWarning)

# ── Path resolution ──────────────────────────────────────────────────────────
# Override by setting DATA_DIR env variable, e.g.  export DATA_DIR=/path/to/data
ROOT     = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = Path(os.environ.get("DATA_DIR", ROOT / "data"))

print(f"DATA_DIR: {DATA_DIR}")
assert DATA_DIR.exists(), f"Data directory not found: {DATA_DIR}"

## 2. Load raw data

In [ ]:
CSV_FILES = [f"{yr}CostReport.csv" for yr in range(2015, 2022)]

dfs = []
for fname in CSV_FILES:
    fpath = DATA_DIR / fname
    df = pd.read_csv(fpath, low_memory=False)
    df["year"] = fname[:4]          # add year column before concat (avoids fragmentation warning)
    dfs.append(df)
    print(f"Loaded {fname}: {df.shape}")

raw = pd.concat(dfs, ignore_index=True)
print(f"
Merged shape: {raw.shape}")

## 3. Shape & basic info

In [ ]:
print("Shape:", raw.shape)
print("
Dtypes summary:")
print(raw.dtypes.value_counts())
print("
First 3 rows:")
raw.head(3)

## 4. Missing value analysis

In [ ]:
missing = raw.isnull().sum()
missing_pct = (missing / len(raw) * 100).round(2)
missing_report = (
    pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
    .query("missing_count > 0")
    .sort_values("missing_pct", ascending=False)
)
print(f"Columns with missing values: {len(missing_report)}")
missing_report.head(20)

In [ ]:
plt.figure(figsize=(14, 6))
sns.heatmap(raw.isnull(), cbar=False, yticklabels=False)
plt.title("Missing value heatmap (each row = one record)")
plt.tight_layout()
plt.show()

## 5. Categorical column cardinality

In [ ]:
cat_cols = raw.select_dtypes(include=["object", "category", "str"]).columns.tolist()
print("Categorical columns:", cat_cols)
for col in cat_cols:
    print(f"  {col}: {raw[col].nunique()} unique values")

## 6. Numerical column distributions

In [ ]:
num_cols = raw.select_dtypes(include=["int64", "float64"]).columns.tolist()

def plot_histogram(df, col, bins=100, xlim=None, color="steelblue"):
    """Reusable histogram helper with optional x-axis clipping."""
    fig, ax = plt.subplots(figsize=(8, 3))
    sns.histplot(df[col].dropna(), bins=bins, color=color, kde=True, ax=ax)
    ax.set_title(f"Distribution of {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Frequency")
    if xlim:
        ax.set_xlim(xlim)
    plt.tight_layout()
    plt.show()

# Overview histogram grid for all numerical columns
num_data = raw[num_cols]
num_rows = int(np.ceil(len(num_cols) / 5))
num_data.hist(bins=30, figsize=(18, num_rows * 3), layout=(num_rows, 5))
plt.suptitle("Numerical column distributions (raw data)", y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

## 7. Year-level trends

In [ ]:
records_by_year = raw.groupby("year").size().reset_index(name="count")
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=records_by_year, x="year", y="count", color="steelblue", ax=ax)
ax.set_title("Number of facilities per year")
ax.set_xlabel("Year")
ax.set_ylabel("Facility count")
plt.tight_layout()
plt.show()

## 8. State-level distribution

In [ ]:
# Normalise state column name defensively
state_col = next((c for c in raw.columns if "state" in c.lower() and "code" in c.lower()), None)
if state_col:
    top_states = raw[state_col].value_counts().head(20)
    fig, ax = plt.subplots(figsize=(12, 4))
    top_states.plot(kind="bar", color="steelblue", ax=ax)
    ax.set_title("Top 20 states by facility count")
    ax.set_ylabel("Count")
    ax.set_xlabel("State")
    plt.tight_layout()
    plt.show()
else:
    print("State code column not found — check column names.")

## 9. Rural vs urban breakdown

In [ ]:
rural_col = next((c for c in raw.columns if "rural" in c.lower()), None)
if rural_col:
    print(raw[rural_col].value_counts())
    fig, ax = plt.subplots(figsize=(5, 4))
    raw[rural_col].value_counts().plot(kind="bar", color=["#4C9BE8", "#E87C4C"], ax=ax)
    ax.set_title("Rural vs urban facility count")
    ax.set_xlabel("Category")
    ax.set_ylabel("Count")
    plt.tight_layout()
    plt.show()
else:
    print("Rural/urban column not found.")

## 10. Correlation heatmap

> Note: computed on raw data before imputation — missing values are excluded per-pair.

In [ ]:
numerical_data = raw.select_dtypes(include=["int64", "float64"])
correlation_matrix = numerical_data.corr()
tri_mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(
    correlation_matrix,
    mask=tri_mask,
    cmap="coolwarm",
    linewidths=0.3,
    square=True,
    vmin=-1, vmax=1,
    cbar_kws={"shrink": 0.7},
    ax=ax
)
ax.set_title("Correlation matrix (lower triangle, raw numerical columns)")
plt.tight_layout()
plt.show()

## 11. Pre/post COVID comparison (2019 vs 2020–2021)

Compares mean values of key financial columns across the pre-COVID and COVID periods.

In [ ]:
# Columns to compare — adjust as needed
financial_cols = [c for c in num_cols if any(k in c.lower() for k in
    ["income", "revenue", "cost", "asset", "liabilit", "days", "beds"])][:8]

raw["period"] = raw["year"].apply(lambda y: "Post-COVID (2020–21)" if int(y) >= 2020 else "Pre-COVID (2015–19)")

comparison = (
    raw.groupby("period")[financial_cols]
    .mean()
    .T
    .rename_axis("column")
    .reset_index()
)

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(financial_cols):
    subset = raw.groupby("period")[col].mean().reset_index()
    axes[i].bar(subset["period"], subset[col], color=["#4C9BE8", "#E87C4C"])
    axes[i].set_title(col[:30], fontsize=9)
    axes[i].tick_params(axis="x", labelrotation=15, labelsize=7)

plt.suptitle("Mean financial indicators: pre vs post COVID", fontsize=13)
plt.tight_layout()
plt.show()